# EOM SFT Fine-tuning — Gemma-2-2B (P100/T4/V100 compatible)

**Objective**: Fine-tune a LoRA adapter on the EOM SFT dataset so a small local Gemma model can compile source documents into EOM JSON without calling OpenRouter.

## Dataset stats
| Split | Examples |
|-------|----------|
| train (sft.jsonl) | 104 |
| val   (val.jsonl) | 5 |
| test  (test.jsonl) | 7 |

Token lengths (combined input+target): median 3.8K, p95 8.7K, max 12K. `max_seq_length=4096` filters out ~10% of training examples that exceed 4K.

## Why no Unsloth
Kaggle's free GPU pool serves both **P100** (Pascal sm_60) and **T4** (Turing sm_75). Modern PyTorch (≥2.5) and Unsloth dropped sm_60 support, so any Pascal allocation crashes with `cudaErrorNoKernelImageForDevice`. This notebook uses vanilla `transformers + peft + trl` in fp16, which works on every GPU Kaggle assigns.

## Output
After ~30-60 min the notebook saves:
- `/kaggle/working/eom-sft-adapter/` — LoRA adapter (~30 MB, download from Output tab)
- `/kaggle/working/val-generations.jsonl` — greedy decodes on val for offline harness eval

In [ ]:
# Install/upgrade only what we need on top of Kaggle's pre-installed torch.
# Pin transformers to a version with stable Gemma chat-template support.
!pip install -q -U "transformers>=4.45,<5" "peft>=0.12" "trl>=0.10" "datasets>=2.20" "accelerate>=0.34"

In [ ]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
    cap = torch.cuda.get_device_capability(0)
    print(f"Compute capability: sm_{cap[0]}{cap[1]}")

In [ ]:
# --- Model choice ---
# Default: Gemma-2-2B instruction-tuned, fp16. ~5 GB VRAM for weights.
# With LoRA + seq=4096 + batch=1 + grad-checkpointing, total VRAM is ~10-12 GB.
# Fits comfortably on P100 (16GB), T4 (16GB), V100 (16/32GB).
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "google/gemma-2-2b-it"
max_seq_length = 4096

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager",  # Pascal/Turing-safe; no flash-attention on these GPUs
)
model.config.use_cache = False  # required for gradient checkpointing
model.gradient_checkpointing_enable()
print("Loaded:", model_name)

In [ ]:
# Apply LoRA (rank 16, alpha 32 on attention + MLP projections).
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# We're not using k-bit quantization; still call prepare_* to enable
# input gradients for gradient checkpointing.
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

TRAIN_PATH = "/kaggle/input/eom-sft/sft.jsonl"
VAL_PATH   = "/kaggle/input/eom-sft/val.jsonl"

raw = load_dataset("json", data_files={"train": TRAIN_PATH, "val": VAL_PATH})
print("Raw:", {k: len(v) for k, v in raw.items()})


def to_chat(ex):
    """Wrap (input, target) in the Gemma chat template."""
    messages = [
        {"role": "user",      "content": ex["input"]},
        {"role": "assistant", "content": ex["target"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}


dataset = raw.map(to_chat, remove_columns=raw["train"].column_names)


def fits(ex):
    ids = tokenizer(ex["text"], add_special_tokens=False)["input_ids"]
    return len(ids) <= max_seq_length


dataset = dataset.filter(fits)
print("After length filter:", {k: len(v) for k, v in dataset.items()})

In [ ]:
from trl import SFTTrainer, SFTConfig

# Effective batch = per_device (1) × grad_accum (16) = 16.
args = SFTConfig(
    output_dir="/kaggle/working/eom-sft-out",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_torch",  # NOT adamw_8bit — bnb 8-bit kernels need sm_70+
    max_seq_length=max_seq_length,
    packing=False,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    bf16=False,
    seed=42,
    report_to="none",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    dataset_text_field="text",
    args=args,
)
n_train = len(dataset["train"])
eff_batch = args.per_device_train_batch_size * args.gradient_accumulation_steps
print(f"Steps per epoch: {n_train // eff_batch} (train={n_train}, eff_batch={eff_batch})")

In [ ]:
stats = trainer.train()
print("Training complete.")
print(f"  Peak VRAM: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")
print(f"  Training loss: {stats.training_loss:.4f}")

In [ ]:
# --- Validation generations ---
# Generate greedy decodes per val example so harness eval can run offline.
import json

model.eval()
gens = []
for ex in dataset["val"]:
    prompt_text = ex["text"].split("<start_of_turn>model")[0] + "<start_of_turn>model\n"
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    gens.append({"prompt": prompt_text, "generated": generated})

out_path = "/kaggle/working/val-generations.jsonl"
with open(out_path, "w") as f:
    for g in gens:
        f.write(json.dumps(g) + "\n")
print(f"Saved {len(gens)} val generations -> {out_path}")

In [ ]:
import os

ADAPTER_PATH = "/kaggle/working/eom-sft-adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to {ADAPTER_PATH}")
print("Files:", os.listdir(ADAPTER_PATH))

## Using the adapter locally with FineTunedCompiler

Download `eom-sft-adapter/` from the Kaggle Output tab, then:

```bash
export EOM_FINETUNED_CKPT="/path/to/eom-sft-adapter"
uv sync --extra ml   # installs torch, transformers, peft
uv run python -c "
from eom.compilers.finetuned import FineTunedCompiler
c = FineTunedCompiler()
eom = c.compile(open('tests/fixtures/freight_memo.md').read(), hints={'document_type': 'memo'})
print(eom.model_dump_json(indent=2)[:500])
"
```